<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 6: RAG Context Builder

이번 실습에서는 검색된 여러 문서 조각(chunk) 중에서
**질문에 필요한 근거를 골라 LLM에 전달할 Context를 만드는 과정**을 구현합니다.

### 학습 목표
- 검색 결과(Retrieval Result)와 최종 Context의 차이를 이해한다.
- 관련성이 낮은 chunk를 제거한다.
- 중복 chunk를 제거한다.
- Context 길이 예산(Context Budget)을 적용한다.
- 출처 정보를 포함한 RAG Prompt를 만든다.

> 핵심: **검색된 문서를 모두 넣는 것이 아니라, 필요한 근거만 골라 Context를 구성합니다.**

## 1. 필요한 라이브러리

이번 실습은 Colab 기본 환경에서 바로 실행할 수 있습니다.

In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. 실습용 문서 Chunk 준비

하나의 강의 안내 문서가 여러 chunk로 나뉘어 있다고 가정합니다.

각 chunk에는:
- `chunk_id`
- `source`
- `text`

정보가 있습니다.

In [2]:
chunks = [
    {
        "chunk_id": "c1",
        "source": "수업계획서",
        "text": "지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다."
    },
    {
        "chunk_id": "c2",
        "source": "수업계획서",
        "text": "성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다."
    },
    {
        "chunk_id": "c3",
        "source": "출석안내",
        "text": "출석 점수는 전체 성적의 20%이다."
    },
    {
        "chunk_id": "c4",
        "source": "실습안내",
        "text": "과제와 실습은 Google Colab 환경에서 Python으로 수행한다."
    },
    {
        "chunk_id": "c5",
        "source": "RAG강의자료",
        "text": "RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 방법을 학습한다."
    },
    {
        "chunk_id": "c6",
        "source": "LMS안내",
        "text": "강의 자료와 실습 파일은 LMS를 통해 제공한다."
    },
    {
        "chunk_id": "c7",
        "source": "수업계획서",
        "text": "과제 반영 비율은 전체 성적의 20%이다."
    }
]

for item in chunks:
    print(
        item["chunk_id"],
        "|",
        item["source"],
        "|",
        item["text"]
    )

c1 | 수업계획서 | 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.
c2 | 수업계획서 | 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.
c3 | 출석안내 | 출석 점수는 전체 성적의 20%이다.
c4 | 실습안내 | 과제와 실습은 Google Colab 환경에서 Python으로 수행한다.
c5 | RAG강의자료 | RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 방법을 학습한다.
c6 | LMS안내 | 강의 자료와 실습 파일은 LMS를 통해 제공한다.
c7 | 수업계획서 | 과제 반영 비율은 전체 성적의 20%이다.


## 3. 질문 준비

사용자 질문은 다음과 같습니다.

**“지능시스템 과목의 과제 비율은 몇 퍼센트인가?”**

정답 근거는 `c2` 또는 `c7`에 들어 있습니다.

In [3]:
query = "지능시스템 과목의 과제 비율은 몇 퍼센트인가?"

print("질문:", query)

질문: 지능시스템 과목의 과제 비율은 몇 퍼센트인가?


## 4. 간단한 검색기 만들기

검색 과정은 TF-IDF + Cosine Similarity로 구현합니다.

이 실습의 핵심은 검색 알고리즘 자체보다
**검색 결과를 Context로 구성하는 과정**입니다.

In [4]:
def tokenize_korean(text):
    """
    간단한 한국어 tokenizer

    1. 영문은 소문자로 바꿉니다.
    2. 한글, 영문, 숫자, %만 남깁니다.
    3. 공백으로 단어를 나눕니다.
    """

    text = text.lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣%]+", " ", text)

    return text.split()


# 검색용 문서 목록
texts = [item["text"] for item in chunks]

# TF-IDF 모델 생성
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_korean,
    token_pattern=None,
    lowercase=False
)

# 문서들을 TF-IDF 벡터로 변환
X = vectorizer.fit_transform(texts)

print("문서 수:", X.shape[0])
print("TF-IDF 차원:", X.shape[1])

문서 수: 7
TF-IDF 차원: 47


In [5]:
def retrieve(query, top_k=5):
    """
    질문과 관련된 상위 top_k chunk를 반환합니다.
    """

    # 질문을 TF-IDF 벡터로 변환
    q = vectorizer.transform([query])

    # 질문과 각 chunk의 cosine similarity 계산
    scores = cosine_similarity(q, X)[0]

    # 점수가 높은 순서대로 정렬
    ranking = np.argsort(-scores)

    results = []

    for idx in ranking[:top_k]:
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(scores[idx])
        })

    return results

## 5. Top-k 검색 결과 확인

In [6]:
retrieved = retrieve(
    query,
    top_k=5
)

for rank, item in enumerate(retrieved, 1):
    print(
        f"{rank}위",
        item["chunk_id"],
        f"score={item['score']:.4f}"
    )
    print("출처:", item["source"])
    print("내용:", item["text"])
    print()

1위 c7 score=0.4723
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

2위 c1 score=0.2069
출처: 수업계획서
내용: 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.

3위 c2 score=0.1194
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.

4위 c3 score=0.0000
출처: 출석안내
내용: 출석 점수는 전체 성적의 20%이다.

5위 c4 score=0.0000
출처: 실습안내
내용: 과제와 실습은 Google Colab 환경에서 Python으로 수행한다.



## 6. 검색 결과를 모두 Context에 넣어 보기

가장 단순한 방법은 Top-k 결과를 모두 이어 붙이는 것입니다.

하지만 이 방식은:
- 질문과 관계없는 내용
- 중복된 내용
- 너무 긴 Context

를 포함할 수 있습니다.

In [7]:
def build_naive_context(retrieved_results):
    """
    검색된 모든 chunk를 그대로 Context에 넣습니다.
    """

    parts = []

    for item in retrieved_results:
        parts.append(
            f"[{item['chunk_id']} | {item['source']}]\n"
            f"{item['text']}"
        )

    return "\n\n".join(parts)


naive_context = build_naive_context(retrieved)

print("=== Naive Context ===")
print(naive_context)

=== Naive Context ===
[c7 | 수업계획서]
과제 반영 비율은 전체 성적의 20%이다.

[c1 | 수업계획서]
지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.

[c2 | 수업계획서]
성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.

[c3 | 출석안내]
출석 점수는 전체 성적의 20%이다.

[c4 | 실습안내]
과제와 실습은 Google Colab 환경에서 Python으로 수행한다.


# 7. Context Builder ① 관련성이 낮은 Chunk 제거

검색 점수가 너무 낮은 chunk는 Context에서 제외합니다.

여기서는 간단히 `min_score` 기준을 사용합니다.

> 실제 시스템에서는 reranker나 relevance classifier 등을 함께 사용할 수 있습니다.

In [8]:
def filter_by_score(
    retrieved_results,
    min_score=0.05
):
    """
    검색 점수가 min_score 이상인 chunk만 남깁니다.
    """

    selected = []

    for item in retrieved_results:

        if item["score"] >= min_score:
            selected.append(item)

    return selected


filtered = filter_by_score(
    retrieved,
    min_score=0.05
)

for item in filtered:
    print(
        item["chunk_id"],
        f"{item['score']:.4f}",
        item["text"]
    )

c7 0.4723 과제 반영 비율은 전체 성적의 20%이다.
c1 0.2069 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.
c2 0.1194 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.


# 8. Context Builder ② 중복 Chunk 제거

RAG에서는 같은 내용을 표현하는 chunk가 여러 개 검색될 수 있습니다.

현재 예제에서는:

- `c2`: 과제 20%
- `c7`: 과제 반영 비율 20%

처럼 의미가 매우 비슷한 근거가 있습니다.

간단한 방법으로 **chunk 간 cosine similarity가 너무 높으면 중복으로 판단**해 보겠습니다.

In [9]:
def remove_duplicates(
    selected_results,
    similarity_threshold=0.70
):
    """
    의미가 매우 비슷한 chunk를 하나만 남깁니다.

    이미 선택한 chunk와 유사도가 threshold 이상이면
    중복으로 보고 제외합니다.
    """

    if not selected_results:
        return []

    kept = []

    for item in selected_results:

        # 아직 아무것도 없으면 첫 chunk는 저장
        if len(kept) == 0:
            kept.append(item)
            continue

        # 현재 chunk를 벡터로 변환
        current_vec = vectorizer.transform(
            [item["text"]]
        )

        is_duplicate = False

        for kept_item in kept:

            kept_vec = vectorizer.transform(
                [kept_item["text"]]
            )

            similarity = cosine_similarity(
                current_vec,
                kept_vec
            )[0][0]

            # 너무 비슷하면 중복
            if similarity >= similarity_threshold:
                is_duplicate = True
                break

        # 중복이 아니면 저장
        if not is_duplicate:
            kept.append(item)

    return kept

In [10]:
deduplicated = remove_duplicates(
    filtered,
    similarity_threshold=0.70
)

print("=== 중복 제거 후 ===")

for item in deduplicated:
    print(
        item["chunk_id"],
        f"{item['score']:.4f}",
        item["text"]
    )

=== 중복 제거 후 ===
c7 0.4723 과제 반영 비율은 전체 성적의 20%이다.
c1 0.2069 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.
c2 0.1194 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.


# 9. Context Builder ③ Context Budget 적용

LLM에는 입력 길이 제한이 있기 때문에
검색 결과를 무한정 넣을 수 없습니다.

이번 실습에서는 이해하기 쉽게
**최대 글자 수(max_chars)** 를 Context Budget으로 사용합니다.

실제 시스템에서는 보통 tokenizer를 이용해 **token 수**로 관리합니다.

In [11]:
def apply_context_budget(
    selected_results,
    max_chars=180
):
    """
    Context의 최대 글자 수를 넘지 않도록
    상위 chunk부터 선택합니다.
    """

    final_chunks = []
    used_chars = 0

    for item in selected_results:

        text_length = len(item["text"])

        # 이 chunk를 추가했을 때 예산을 넘으면 제외
        if used_chars + text_length > max_chars:
            continue

        final_chunks.append(item)
        used_chars += text_length

    return final_chunks

In [12]:
budgeted = apply_context_budget(
    deduplicated,
    max_chars=180
)

print("=== Context Budget 적용 후 ===")

for item in budgeted:
    print(
        item["chunk_id"],
        "| 글자 수:",
        len(item["text"]),
        "|",
        item["text"]
    )

=== Context Budget 적용 후 ===
c7 | 글자 수: 23 | 과제 반영 비율은 전체 성적의 20%이다.
c1 | 글자 수: 43 | 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.
c2 | 글자 수: 48 | 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.


# 10. 최종 Context 만들기

최종 Context에는:
- chunk 번호
- 출처
- 실제 근거 내용

을 함께 넣습니다.

출처 정보를 함께 넣으면 나중에 Citation을 만들기 쉽습니다.

In [13]:
def format_context(selected_results):
    """
    선택된 chunk를 LLM에 전달할 Context 형식으로 만듭니다.
    """

    parts = []

    for item in selected_results:

        part = (
            f"[{item['chunk_id']}]\n"
            f"출처: {item['source']}\n"
            f"내용: {item['text']}"
        )

        parts.append(part)

    return "\n\n".join(parts)


final_context = format_context(budgeted)

print("=== Final Context ===")
print(final_context)

=== Final Context ===
[c7]
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

[c1]
출처: 수업계획서
내용: 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.

[c2]
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.


# 11. 전체 Context Builder 함수 만들기

지금까지의 과정을 하나의 함수로 연결합니다.

**Retrieve → Score Filter → Dedup → Budget → Context**

In [14]:
def context_builder(
    query,
    top_k=5,
    min_score=0.05,
    duplicate_threshold=0.70,
    max_chars=180
):
    """
    RAG Context Builder

    1. 관련 문서를 검색
    2. 낮은 점수 제거
    3. 중복 제거
    4. Context 길이 제한
    5. 최종 Context 생성
    """

    # 1. 검색
    retrieved = retrieve(
        query,
        top_k=top_k
    )

    # 2. 관련성이 낮은 chunk 제거
    filtered = filter_by_score(
        retrieved,
        min_score=min_score
    )

    # 3. 중복 제거
    deduplicated = remove_duplicates(
        filtered,
        similarity_threshold=duplicate_threshold
    )

    # 4. Context Budget 적용
    selected = apply_context_budget(
        deduplicated,
        max_chars=max_chars
    )

    # 5. 최종 문자열 생성
    context = format_context(selected)

    return selected, context

In [15]:
selected, final_context = context_builder(
    query
)

print("질문:")
print(query)

print("\n선택된 Chunk:")
for item in selected:
    print(
        item["chunk_id"],
        f"score={item['score']:.4f}"
    )

print("\n최종 Context:")
print(final_context)

질문:
지능시스템 과목의 과제 비율은 몇 퍼센트인가?

선택된 Chunk:
c7 score=0.4723
c1 score=0.2069
c2 score=0.1194

최종 Context:
[c7]
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

[c1]
출처: 수업계획서
내용: 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.

[c2]
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.


# 12. RAG Prompt 만들기

이제 Context를 LLM에 전달할 Prompt를 만듭니다.

중요한 규칙:
1. 제공된 Context만 사용
2. 근거가 없으면 추측하지 않기
3. 출처 chunk 번호 표시

In [16]:
def make_rag_prompt(
    query,
    context
):
    """
    Context와 질문을 이용해
    RAG Prompt를 만듭니다.
    """

    prompt = f"""
아래 Context만 사용하여 질문에 답하세요.

규칙:
1. Context에 없는 정보는 추측하지 마세요.
2. 답을 확인할 수 없으면
   "제공된 자료에서 확인할 수 없습니다."라고 답하세요.
3. 답변 뒤에 근거 chunk 번호를 표시하세요.

[Context]
{context}

[Question]
{query}

[Answer]
"""

    return prompt.strip()

In [17]:
prompt = make_rag_prompt(
    query,
    final_context
)

print(prompt)

아래 Context만 사용하여 질문에 답하세요.

규칙:
1. Context에 없는 정보는 추측하지 마세요.
2. 답을 확인할 수 없으면
   "제공된 자료에서 확인할 수 없습니다."라고 답하세요.
3. 답변 뒤에 근거 chunk 번호를 표시하세요.

[Context]
[c7]
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

[c1]
출처: 수업계획서
내용: 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.

[c2]
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.

[Question]
지능시스템 과목의 과제 비율은 몇 퍼센트인가?

[Answer]


## 13. 예상 답변 확인

현재 질문에 대한 좋은 답변은 다음과 같습니다.

> **과제 비율은 20%입니다. [c2]**

핵심은 LLM이 답을 만들어내기 전에
정확한 근거가 Context에 들어 있어야 한다는 점입니다.

In [18]:
expected_answer = "과제 비율은 20%입니다. [c2]"

print("예상 답변:")
print(expected_answer)

예상 답변:
과제 비율은 20%입니다. [c2]


# 14. 문서에 없는 질문 처리

이번에는 문서에 없는 질문을 사용합니다.

**“지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?”**

정답 근거가 없다면 Context Builder도
신뢰할 수 있는 근거를 제공할 수 없어야 합니다.

In [21]:
unknown_query = "지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?"

unknown_selected, unknown_context = context_builder(
    unknown_query,
    min_score=0.40
)

print("질문:")
print(unknown_query)

print("\n선택된 Chunk:")
for item in unknown_selected:
    print(
        item["chunk_id"],
        f"score={item['score']:.4f}",
        item["text"]
    )

print("\nContext:")
print(
    unknown_context
    if unknown_context
    else "신뢰할 수 있는 근거 없음"
)

질문:
지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?

선택된 Chunk:

Context:
신뢰할 수 있는 근거 없음


### 주의

단순한 TF-IDF 점수 기준만으로
“답할 수 있는 질문 / 답할 수 없는 질문”을 완벽하게 구분할 수는 없습니다.

실제 RAG에서는 다음을 추가로 사용할 수 있습니다.

- Reranker
- relevance classifier
- LLM judge
- confidence threshold
- answerability check

따라서 이번 threshold는 **교육용 단순화**입니다.

# 15. 여러 질문으로 Context Builder 테스트

In [22]:
test_queries = [
    "과제 비율은 얼마인가?",
    "출석 비율은 얼마인가?",
    "실습은 어떤 환경에서 하는가?",
    "RAG 실습에서는 무엇을 배우는가?"
]

for q in test_queries:

    selected, context = context_builder(q)

    print("=" * 70)
    print("질문:", q)

    print("\n선택된 chunk:")
    print(
        [item["chunk_id"] for item in selected]
    )

    print("\nContext:")
    print(context)
    print()

질문: 과제 비율은 얼마인가?

선택된 chunk:
['c7', 'c2']

Context:
[c7]
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

[c2]
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.

질문: 출석 비율은 얼마인가?

선택된 chunk:
['c7', 'c3', 'c2']

Context:
[c7]
출처: 수업계획서
내용: 과제 반영 비율은 전체 성적의 20%이다.

[c3]
출처: 출석안내
내용: 출석 점수는 전체 성적의 20%이다.

[c2]
출처: 수업계획서
내용: 성적 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%로 구성된다.

질문: 실습은 어떤 환경에서 하는가?

선택된 chunk:
['c4']

Context:
[c4]
출처: 실습안내
내용: 과제와 실습은 Google Colab 환경에서 Python으로 수행한다.

질문: RAG 실습에서는 무엇을 배우는가?

선택된 chunk:
['c5', 'c1']

Context:
[c5]
출처: RAG강의자료
내용: RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 방법을 학습한다.

[c1]
출처: 수업계획서
내용: 지능시스템 과목은 자연어처리, RAG, AI 에이전트를 다루는 전공 과목이다.



# 16. 실습 전체 흐름

이번 실습에서 구현한 흐름은 다음과 같습니다.

**Question**
→ **Retrieve Top-k**
→ **Relevance Filter**
→ **Deduplication**
→ **Context Budget**
→ **Context Construction**
→ **Prompt Construction**

즉, 검색 결과와 LLM 입력 사이에
**Context Builder**가 존재합니다.

# 핵심 정리

### Retriever
질문과 관련된 **후보 문서**를 찾습니다.

### Context Builder
후보 문서 중에서 **LLM이 실제로 사용할 근거**를 선택하고 정리합니다.

### 좋은 Context의 조건
- 질문과 관련 있음
- 중복이 적음
- 중요한 근거가 포함됨
- 입력 길이 예산을 넘지 않음
- 출처를 추적할 수 있음

> **Retrieved ≠ Context**  
> 검색되었다고 해서 모든 문서를 LLM에 넣어야 하는 것은 아닙니다.